# Thí nghiệm CNN-only với preprocessing mới

Notebook này chạy baseline CNN-only bằng pipeline chung trong `preprocessing/preprocess_data.py`.

- CSIC chỉ giữ giá trị query/body, bỏ URL path và tên tham số.
- Không URL decode, không HTML unescape, không lowercase.
- Tokenizer chỉ fit trên train.
- Điểm thay đổi kiến trúc duy nhất so với CNN-LSTM nối tiếp là thay LSTM bằng `GlobalMaxPooling1D`.

In [1]:
from pathlib import Path
import os

START_DIR = Path.cwd().resolve()
if (START_DIR / 'cnn_only' / 'train_cnn_only.py').exists():
    PROJECT_ROOT = START_DIR
    NOTEBOOK_DIR = PROJECT_ROOT / 'cnn_only'
elif START_DIR.name == 'cnn_only' and (START_DIR / 'train_cnn_only.py').exists():
    NOTEBOOK_DIR = START_DIR
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    raise FileNotFoundError('Hãy chạy notebook từ repo root hoặc thư mục cnn_only.')

os.chdir(NOTEBOOK_DIR)
print('Project root:', PROJECT_ROOT)
print('Notebook dir:', NOTEBOOK_DIR)

Project root: C:\Users\admin\Desktop\obfuscated-web-attack-detection
Notebook dir: C:\Users\admin\Desktop\obfuscated-web-attack-detection\cnn_only


## Smoke test
Chạy trên mẫu nhỏ để kiểm tra pipeline. Kết quả này không dùng cho báo cáo.

In [2]:
%run train_cnn_only.py --sample-size 3000 --obfu-sample-size 1000 --epochs 3 --batch-size 128 --output-dir artifacts_new_preprocessing_smoke

=== CNN-ONLY EXPERIMENT ===
Train: 2,160 | Val: 240 | Test: 600
Obfuscated test: 1,000
X_train: (2160, 1024) | Vocab: 119
Shared preprocessing: C:\Users\admin\Desktop\obfuscated-web-attack-detection\preprocessing\preprocess_data.py
CSIC policy: Use raw query/body parameter values only; drop requests with no input values.


Model: "CNN_Only_Web_Attack_Detector"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ char_embedding (Embedding)      │ (None, 1024, 64)       │         7,616 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k3 (Conv1D)                │ (None, 1024, 128)      │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_1 (MaxPooling1D)           │ (None, 256, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k5 (Conv1D)                │ (None, 256, 128)       │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_2 (MaxPooling1D)           │ (None, 64, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pool                 │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_classifier (Dense)        │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attack_probability (Dense)      │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 122,689 (479.25 KB)

 Trainable params: 122,689 (479.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 316ms/step - accuracy: 0.7622 - auc: 0.7513 - loss: 0.6561 - precision: 0.7806 - recall: 0.8996
Epoch 1: val_loss improved from None to 0.32292, saving model to artifacts_new_preprocessing_smoke\best_cnn_only.keras

Epoch 1: finished saving model to artifacts_new_preprocessing_smoke\best_cnn_only.keras
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 350ms/step - accuracy: 0.8319 - auc: 0.8759 - loss: 0.5852 - precision: 0.8547 - recall: 0.8953 - val_accuracy: 0.9167 - val_auc: 0.9409 - val_loss: 0.3229 - val_precision: 0.9202 - val_recall: 0.9554
Epoch 2/3
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 268ms/step - accuracy: 0.9293 - auc: 0.9569 - loss: 0.2877 - precision: 0.9445 - recall: 0.9468
Epoch 2: val_loss improved from 0.32292 to 0.16301, saving model to artifacts_new_preprocessing_smoke\best_cnn_only.keras

Epoch 2: finished saving model to artifacts_new_preprocessing_smoke\best_cnn_only.keras
17/17 ━━━━━━━━━━━━━━━━━━━━ 5s 284ms/step - accuracy: 0.9255 - auc: 0.9620 - lo

## Huấn luyện đầy đủ
Chỉ chạy cell dưới đây sau khi smoke test thành công.

In [3]:
%run train_cnn_only.py --epochs 50 --batch-size 128 --output-dir artifacts_new_preprocessing

=== CNN-ONLY EXPERIMENT ===
Train: 117,444 | Val: 13,050 | Test: 32,624
Obfuscated test: 150,000
X_train: (117444, 1024) | Vocab: 180
Shared preprocessing: C:\Users\admin\Desktop\obfuscated-web-attack-detection\preprocessing\preprocess_data.py
CSIC policy: Use raw query/body parameter values only; drop requests with no input values.


Model: "CNN_Only_Web_Attack_Detector"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ char_embedding (Embedding)      │ (None, 1024, 64)       │        11,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k3 (Conv1D)                │ (None, 1024, 128)      │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_1 (MaxPooling1D)           │ (None, 256, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k5 (Conv1D)                │ (None, 256, 128)       │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_2 (MaxPooling1D)           │ (None, 64, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pool                 │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_classifier (Dense)        │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attack_probability (Dense)      │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 126,593 (494.50 KB)

 Trainable params: 126,593 (494.50 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
918/918 ━━━━━━━━━━━━━━━━━━━━ 0s 250ms/step - accuracy: 0.9487 - auc: 0.9834 - loss: 0.1366 - precision: 0.9607 - recall: 0.9627
Epoch 1: val_loss improved from None to 0.05234, saving model to artifacts_new_preprocessing\best_cnn_only.keras

Epoch 1: finished saving model to artifacts_new_preprocessing\best_cnn_only.keras
918/918 ━━━━━━━━━━━━━━━━━━━━ 241s 260ms/step - accuracy: 0.9693 - auc: 0.9963 - loss: 0.0732 - precision: 0.9856 - recall: 0.9663 - val_accuracy: 0.9773 - val_auc: 0.9982 - val_loss: 0.0523 - val_precision: 0.9993 - val_recall: 0.9654
Epoch 2/50
918/918 ━━━━━━━━━━━━━━━━━━━━ 0s 279ms/step - accuracy: 0.9793 - auc: 0.9983 - loss: 0.0425 - precision: 0.9969 - recall: 0.9708
Epoch 2: val_loss improved from 0.05234 to 0.04409, saving model to artifacts_new_preprocessing\best_cnn_only.keras

Epoch 2: finished saving model to artifacts_new_preprocessing\best_cnn_only.keras
918/918 ━━━━━━━━━━━━━━━━━━━━ 265s 289ms/step - accuracy: 0.9801 - auc: 0.9984 - loss: 0.0410

In [11]:
# Tune the decision threshold after training, using validation data only.
# Pipeline: train -> tune on validation -> evaluate normal/obfuscated test with the selected threshold.
import argparse
import json
import pickle
import sys
from pathlib import Path

import numpy as np
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from tensorflow.keras.models import load_model

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from cnn_lstm import CNN_LSTM as pipeline

ARTIFACT_DIR = Path('artifacts_new_preprocessing')
METADATA_PATH = ARTIFACT_DIR / 'metadata_and_results.json'
MODEL_PATH = ARTIFACT_DIR / 'last_cnn_only.keras'
TOKENIZER_PATH = ARTIFACT_DIR / 'tokenizer.pkl'

with METADATA_PATH.open(encoding='utf-8') as f:
    metadata = json.load(f)

args = argparse.Namespace(
    kaggle_path=str(PROJECT_ROOT / 'SQLInjection_XSS_MixDataset.1.0.0.csv'),
    csic_path=str(PROJECT_ROOT / 'csic_database.csv'),
    obfuscation_path=str(PROJECT_ROOT / 'obfuscation_dataset_full_with_benign_normal.xlsx'),
    test_size=0.2,
    val_size=0.1,
    seed=pipeline.SEED,
    sample_size=None,
    obfu_sample_size=None,
)
_, val_df, test_df, obfuscated_df, rebuilt_metadata = pipeline.build_datasets(args)

expected = metadata.get('splits', {})
# Only the validation/test splits must match the trained artifact.
# The obfuscated dataset is an external robustness test set and may be replaced without retraining.
for split_name, frame in [('val', val_df), ('test', test_df)]:
    expected_rows = expected.get(split_name, {}).get('rows')
    if expected_rows is not None and expected_rows != len(frame):
        raise ValueError(
            f"Rebuilt {split_name} has {len(frame)} rows, but metadata expects {expected_rows}. "
            'Check whether this artifact came from a sampled smoke run.'
        )
metadata.setdefault('splits', {})['obfuscated_test'] = pipeline.summarize(obfuscated_df)

max_len = metadata['cnn_only_model']['max_len']
with TOKENIZER_PATH.open('rb') as f:
    tokenizer = pickle.load(f)
model = load_model(MODEL_PATH, compile=False)

X_val = pipeline.vectorize(tokenizer, val_df['payload'], max_len)
X_test = pipeline.vectorize(tokenizer, test_df['payload'], max_len)
X_obfuscated = pipeline.vectorize(tokenizer, obfuscated_df['payload'], max_len)
y_val = val_df['label'].to_numpy(dtype=np.int32)
y_test = test_df['label'].to_numpy(dtype=np.int32)
y_obfuscated = obfuscated_df['label'].to_numpy(dtype=np.int32)


def metrics_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    auc_roc = None
    if len(np.unique(y_true)) > 1:
        auc_roc = float(roc_auc_score(y_true, y_prob))
    return {
        'threshold': float(threshold),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'auc_roc': auc_roc,
        'confusion_matrix': confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist(),
        'classification_report': classification_report(
            y_true,
            y_pred,
            labels=[0, 1],
            target_names=['Normal (0)', 'Attack (1)'],
            digits=4,
            zero_division=0,
            output_dict=True,
        ),
    }


def print_result(title, result):
    auc_text = 'n/a' if result['auc_roc'] is None else f"{result['auc_roc']:.4f}"
    print(f"\n=== {title} @ threshold={result['threshold']:.2f} ===")
    print(f"Accuracy: {result['accuracy']:.4f}")
    print(f"AUC-ROC : {auc_text}")
    print('Confusion matrix:')
    print(np.array(result['confusion_matrix']))
    attack = result['classification_report']['Attack (1)']
    normal = result['classification_report']['Normal (0)']
    print(f"Normal recall : {normal['recall']:.4f}")
    print(f"Attack recall : {attack['recall']:.4f}")
    print(f"Attack F1     : {attack['f1-score']:.4f}")

val_prob = model.predict(X_val, batch_size=128).flatten()
test_prob = model.predict(X_test, batch_size=128).flatten()
obfuscated_prob = model.predict(X_obfuscated, batch_size=128).flatten()

thresholds = [round(float(x), 2) for x in np.arange(0.05, 0.96, 0.01)]
threshold_rows = [metrics_at_threshold(y_val, val_prob, t) for t in thresholds]
best = max(
    threshold_rows,
    key=lambda row: (
        row['classification_report']['Attack (1)']['f1-score'],
        row['classification_report']['Attack (1)']['recall'],
        row['classification_report']['Normal (0)']['recall'],
    ),
)
selected_threshold = best['threshold']

normal_default = metrics_at_threshold(y_test, test_prob, 0.5)
normal_tuned = metrics_at_threshold(y_test, test_prob, selected_threshold)
obfuscated_default = metrics_at_threshold(y_obfuscated, obfuscated_prob, 0.5)
obfuscated_tuned = metrics_at_threshold(y_obfuscated, obfuscated_prob, selected_threshold)

print(f'Selected threshold from validation: {selected_threshold:.2f}')
print_result('Normal test default', normal_default)
print_result('Normal test tuned', normal_tuned)
print_result('Obfuscated test default', obfuscated_default)
print_result('Obfuscated test tuned', obfuscated_tuned)

metadata['thresholding'] = {
    'mode': 'notebook_tuned_f1',
    'default_threshold': 0.5,
    'selected_threshold': selected_threshold,
    'selection_data': 'validation',
    'selection_rule': 'Maximize validation Attack (1) F1; tie-break by Attack (1) recall, then Normal (0) recall.',
}
metadata['threshold_search_val'] = threshold_rows
metadata.setdefault('evaluation', {})['normal_test_default_0_5'] = normal_default
metadata['evaluation']['normal_test_tuned_threshold'] = normal_tuned
metadata['evaluation']['test_default_0_5'] = normal_default
metadata['evaluation']['test_tuned_threshold'] = normal_tuned
metadata['evaluation']['obfuscated_test_default_0_5'] = obfuscated_default
metadata['evaluation']['obfuscated_test_tuned_threshold'] = obfuscated_tuned

with METADATA_PATH.open('w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print('Saved tuned-threshold metrics to:', METADATA_PATH)


102/102 ━━━━━━━━━━━━━━━━━━━━ 21s 198ms/step
255/255 ━━━━━━━━━━━━━━━━━━━━ 36s 139ms/step
2344/2344 ━━━━━━━━━━━━━━━━━━━━ 451s 192ms/step
Selected threshold from validation: 0.55

=== Normal test default @ threshold=0.50 ===
Accuracy: 0.9836
AUC-ROC : 0.9989
Confusion matrix:
[[11603    63]
 [  471 20487]]
Normal recall : 0.9946
Attack recall : 0.9775
Attack F1     : 0.9871

=== Normal test tuned @ threshold=0.55 ===
Accuracy: 0.9834
AUC-ROC : 0.9989
Confusion matrix:
[[11614    52]
 [  490 20468]]
Normal recall : 0.9955
Attack recall : 0.9766
Attack F1     : 0.9869

=== Obfuscated test default @ threshold=0.50 ===
Accuracy: 0.5622
AUC-ROC : 0.8998
Confusion matrix:
[[ 18767 131233]
 [   105 149895]]
Normal recall : 0.1251
Attack recall : 0.9993
Attack F1     : 0.6954

=== Obfuscated test tuned @ threshold=0.55 ===
Accuracy: 0.5623
AUC-ROC : 0.8998
Confusion matrix:
[[ 18785 131215]
 [   110 149890]]
Normal recall : 0.1252
Attack recall : 0.9993
Attack F1     : 0.6954
Saved tuned-threshol

## Đọc kết quả đã lưu

In [12]:
import json

with open('artifacts_new_preprocessing/metadata_and_results.json', encoding='utf-8') as f:
    results = json.load(f)

print(results['preprocessing_policy'])
print(results['splits']['base_clean'])
print('Selected threshold:', results.get('thresholding', {}).get('selected_threshold', 'Run the threshold tuning cell first.'))
results['cnn_only_model'], results['evaluation']


{'url_decode': False, 'html_unescape': False, 'lowercase': False, 'whitespace_normalization_only': True, 'csic_payload_policy': 'Use raw query/body parameter values only; drop requests with no input values.', 'drop_label_conflicts_base': True, 'drop_label_conflicts_obfuscated_test': False, 'tokenizer_rule': 'Tokenizer is fit on train split only.'}
{'rows': 163118, 'label_counts': {'0': 58327, '1': 104791}, 'source_counts': {'kaggle_sqli_xss': 151662, 'csic_2010': 11456}, 'length': {'mean': 338.49788496671124, 'median': 233.0, 'p90': 798.0, 'p95': 891.0, 'p99': 977.0, 'max': 8493}, 'obfuscation_counts': {'original': 163118}}
Selected threshold: 0.55


({'architecture': 'Embedding -> Conv1D(k3) -> Pool(4) -> Conv1D(k5) -> Pool(4) -> GlobalMaxPooling1D -> Dense -> Sigmoid',
  'max_len': 1024,
  'embedding_dim': 64,
  'vocab_size': 180,
  'parameter_count': 126593,
  'class_weight': {'0': 1.3983093225383973, '1': 0.7783005738975997},
  'training_seconds': 2885.137269400002,
  'epochs_run': 11,
  'seconds_per_epoch_average': 262.2852063090911},
 {'normal_test': {'accuracy': 0.9836316821971555,
   'auc_roc': 0.9989390768344097,
   'confusion_matrix': [[11603, 63], [471, 20487]],
   'classification_report': {'Normal (0)': {'precision': 0.9609905582242836,
     'recall': 0.9945996914109377,
     'f1-score': 0.9775063184498737,
     'support': 11666.0},
    'Attack (1)': {'precision': 0.9969343065693431,
     'recall': 0.9775264815344976,
     'f1-score': 0.9871350101185313,
     'support': 20958.0},
    'accuracy': 0.9836316821971555,
    'macro avg': {'precision': 0.9789624323968134,
     'recall': 0.9860630864727177,
     'f1-score': 0.9

## So sánh CNN-only ở hai ngưỡng
Bảng này so sánh threshold mặc định `0.5` với threshold được tune trên validation set cho cùng một mô hình CNN-only.


In [13]:
import pandas as pd

cnn_eval = results['evaluation']
thresholding = results.get('thresholding', {})
normal_default = cnn_eval.get('normal_test_default_0_5', cnn_eval['normal_test'])
normal_tuned = cnn_eval.get('normal_test_tuned_threshold', cnn_eval['normal_test'])
obfu_default = cnn_eval.get('obfuscated_test_default_0_5', cnn_eval['obfuscated_test'])
obfu_tuned = cnn_eval.get('obfuscated_test_tuned_threshold', cnn_eval['obfuscated_test'])


def attack_metric(result, metric):
    return result['classification_report']['Attack (1)'][metric]


def obfu_recall(result):
    report = result['classification_report']
    if 'Attack (1)' in report:
        return report['Attack (1)']['recall']
    return report['1']['recall']


def missed_attacks(result):
    cm = result['confusion_matrix']
    return cm[1][0] if len(cm) > 1 else None

threshold_comparison = pd.DataFrame([
    {
        'Threshold Mode': 'Default 0.5',
        'Threshold': 0.5,
        'Normal Test Accuracy': normal_default['accuracy'],
        'Normal Test AUC': normal_default['auc_roc'],
        'Attack Recall': attack_metric(normal_default, 'recall'),
        'Attack F1': attack_metric(normal_default, 'f1-score'),
        'Obfuscated Recall': obfu_recall(obfu_default),
        'Obfuscated Missed': missed_attacks(obfu_default),
    },
    {
        'Threshold Mode': 'Tuned on validation',
        'Threshold': thresholding.get('selected_threshold', 0.5),
        'Normal Test Accuracy': normal_tuned['accuracy'],
        'Normal Test AUC': normal_tuned['auc_roc'],
        'Attack Recall': attack_metric(normal_tuned, 'recall'),
        'Attack F1': attack_metric(normal_tuned, 'f1-score'),
        'Obfuscated Recall': obfu_recall(obfu_tuned),
        'Obfuscated Missed': missed_attacks(obfu_tuned),
    },
])
threshold_comparison


,Threshold Mode,Threshold,Normal Test Accuracy,Normal Test AUC,Attack Recall,Attack F1,Obfuscated Recall,Obfuscated Missed
0,Default 0.5,0.50,0.983632,0.998939,0.977526,0.987135,0.999300,105
1,Tuned on validation,0.55,0.983386,0.998939,0.976620,0.986933,0.999267,110
